
# Remaining time by current status — final notebook

Цель: предсказывать **оставшееся время до завершения задачи** в момент входа в текущий статус.

Фиксированный порядок статусов:
`Бэклог → В работе → Ревью → Можно тестировать → Тестируется → Протестировано → Merged → Раскатка`

Что важно:
- `Заблокирован` исключён из milestone-цепочки, потому что это параллельное состояние, а не этап прогресса;
- основная метрика — качество на **post-start** snapshot-данных (`status_idx > 0`);
- backlog оставлен в **supplementary**-анализе: это полезно для полноты картины, но не используется как основной критерий, потому что для backlog задача частично совпадает с полной длительностью от создания;
- split делается **по задачам** и **темпорально**;
- preprocessing, tag selection и любые статистики считаются **только по train**;
- сравнение в финале идёт между:
  - naive baselines,
  - baseline ML,
  - tuned ML.


In [ ]:

import sys, subprocess, importlib, warnings, json, math, time
from pathlib import Path

for pkg in ["optuna", "xgboost"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
from sklearn.linear_model import Lasso, ElasticNet, Ridge, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit

import optuna
from xgboost import XGBRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:

SEED = 2
np.random.seed(SEED)

ART_DIR = Path("./artifacts_remaining_status_final")
ART_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("./data_finall.csv"),
    Path("/mnt/data/data_finall.csv"),
]

DATA_PATH = None
for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate
        break
if DATA_PATH is None:
    raise FileNotFoundError("data_finall.csv not found")

TARGET_COL = "duration_hours"
DATE_FILTER_FROM = "2025-01-01"

STATUS_ORDER = [
    "Бэклог",
    "В работе",
    "Ревью",
    "Можно тестировать",
    "Тестируется",
    "Протестировано",
    "Merged",
    "Раскатка",
]
STATUS_SET = set(STATUS_ORDER)

print("DATA_PATH:", DATA_PATH.resolve())
print("ART_DIR:", ART_DIR.resolve())
print("STATUS_ORDER:", " -> ".join(STATUS_ORDER))


In [ ]:

raw = pd.read_csv(DATA_PATH)

# created_dt
if "created_dt" in raw.columns:
    raw["created_dt"] = pd.to_datetime(raw["created_dt"], errors="coerce")
else:
    raw["created_dt"] = pd.to_datetime(raw["Создано"], errors="coerce")

raw = raw[raw["created_dt"].notna()].copy()
raw = raw[raw["created_dt"] > DATE_FILTER_FROM].copy()
raw = raw.sort_values("created_dt").reset_index(drop=True)

# status duration columns must exist
missing_status_cols = [c for c in STATUS_ORDER if c not in raw.columns]
if missing_status_cols:
    raise ValueError(f"Missing status columns: {missing_status_cols}")

# team label from one-hot columns (for naive baseline StatusTeamMedian)
team_cols = [c for c in raw.columns if c.startswith("Команда_")]
def recover_team_label(row):
    if not team_cols:
        return "unknown"
    vals = row[team_cols].to_numpy(dtype=float)
    if np.nanmax(vals) <= 0:
        return "unknown"
    return team_cols[int(np.nanargmax(vals))].replace("Команда_", "")

raw["team_label"] = raw.apply(recover_team_label, axis=1)

# static feature columns: use already prepared numeric task-level features
drop_static = {
    "Ключ", "Статус", "Резолюция", "Создано", "Дата завершения",
    "created_dt", TARGET_COL,
    "Бэклог", "Заблокирован", "В работе", "Ревью",
    "Можно тестировать", "Тестируется", "Протестировано", "Merged", "Раскатка",
    "team_label",
}
static_feature_cols = [
    c for c in raw.columns
    if c not in drop_static and pd.api.types.is_numeric_dtype(raw[c])
]

print(f"After date filter: {len(raw)} tasks")
print(f"Static numeric features from task-level data: {len(static_feature_cols)}")
print("First static features:", static_feature_cols[:15])


In [ ]:

snapshot_rows = []

for _, task in raw.iterrows():
    total_dur = float(task[TARGET_COL])
    if not np.isfinite(total_dur) or total_dur <= 0:
        continue

    elapsed = 0.0
    for i, status in enumerate(STATUS_ORDER):
        t_in_status = float(task.get(status, 0.0))
        if not np.isfinite(t_in_status) or t_in_status <= 0:
            continue

        remaining = total_dur - elapsed

        row = {
            "task_key": task["Ключ"],
            "created_dt": task["created_dt"],
            "current_status": status,
            "status_idx": i,
            "elapsed": round(elapsed, 6),
            "remaining": round(remaining, 6),
            "team_label": task["team_label"],
        }

        # known past statuses only; current/future -> 0
        for j, s in enumerate(STATUS_ORDER):
            row[f"t_{s}"] = float(task[s]) if j < i else 0.0

        # current status one-hot
        for s in STATUS_ORDER:
            row[f"is_{s}"] = float(s == status)

        # task-level static features
        for c in static_feature_cols:
            row[c] = task[c]

        snapshot_rows.append(row)
        elapsed += t_in_status

snap_df = pd.DataFrame(snapshot_rows)
snap_df = snap_df.sort_values(["created_dt", "task_key", "status_idx"]).reset_index(drop=True)

print(f"Snapshot rows: {len(snap_df)} from {len(raw)} tasks")
print("Rows per task:", round(len(snap_df) / max(len(raw), 1), 2))
print(snap_df["current_status"].value_counts().reindex(STATUS_ORDER).fillna(0).astype(int))


In [ ]:

# Temporal split by task
task_keys = raw["Ключ"].tolist()
split_idx = int(len(task_keys) * 0.8)

train_keys = set(task_keys[:split_idx])
test_keys = set(task_keys[split_idx:])

snap_train_all = snap_df[snap_df["task_key"].isin(train_keys)].copy()
snap_test_all = snap_df[snap_df["task_key"].isin(test_keys)].copy()

# Primary task = post-start snapshots (exclude backlog)
snap_train_ps = snap_train_all[snap_train_all["status_idx"] > 0].copy()
snap_test_ps = snap_test_all[snap_test_all["status_idx"] > 0].copy()

FEATURE_DROP = ["task_key", "created_dt", "current_status", "team_label", "remaining"]

X_all_train = snap_train_all.drop(columns=FEATURE_DROP, errors="ignore")
y_all_train = snap_train_all["remaining"]
X_all_test = snap_test_all.drop(columns=FEATURE_DROP, errors="ignore")
y_all_test = snap_test_all["remaining"]

X_ps_train = snap_train_ps.drop(columns=FEATURE_DROP, errors="ignore")
y_ps_train = snap_train_ps["remaining"]
X_ps_test = snap_test_ps.drop(columns=FEATURE_DROP, errors="ignore")
y_ps_test = snap_test_ps["remaining"]

test_status_names_all = snap_test_all["current_status"].values
test_status_names_ps = snap_test_ps["current_status"].values
test_team_names_ps = snap_test_ps["team_label"].values

print("ALL snapshots:")
print(f"  Train: {len(snap_train_all)} rows ({len(train_keys)} tasks)")
print(f"  Test:  {len(snap_test_all)} rows ({len(test_keys)} tasks)")
print("POST-START snapshots:")
print(f"  Train: {len(snap_train_ps)} rows")
print(f"  Test:  {len(snap_test_ps)} rows")
print(f"  Features: {X_ps_train.shape[1]}")
print(f"  Remaining mean/median (post-start train): {y_ps_train.mean():.1f} / {y_ps_train.median():.1f}")



## Naive baselines and ML baselines

Основная таблица строится на **post-start** snapshot-данных.  
Отдельно в конце будет supplementary-анализ по **всем статусам**, включая backlog.


In [ ]:

def reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

rem_results = []

# Naive baselines
global_med = float(y_ps_train.median())
pred_global = np.full(len(y_ps_test), global_med)
m = reg_metrics(y_ps_test, pred_global)
rem_results.append(dict(model="GlobalMedian", stage="naive", **m))

status_medians = snap_train_ps.groupby("current_status")["remaining"].median().to_dict()
pred_status = np.array([status_medians.get(s, global_med) for s in test_status_names_ps], dtype=float)
m = reg_metrics(y_ps_test, pred_status)
rem_results.append(dict(model="StatusMedian", stage="naive", **m))

status_team_medians = snap_train_ps.groupby(["current_status", "team_label"])["remaining"].median().to_dict()
pred_status_team = np.array([
    status_team_medians.get((s, t), status_medians.get(s, global_med))
    for s, t in zip(test_status_names_ps, test_team_names_ps)
], dtype=float)
m = reg_metrics(y_ps_test, pred_status_team)
rem_results.append(dict(model="StatusTeamMedian", stage="naive", **m))

BASELINE_MODELS = {
    "Lasso": Pipeline([("sc", StandardScaler()), ("m", Lasso(random_state=SEED, max_iter=50000))]),
    "ElasticNet": Pipeline([("sc", StandardScaler()), ("m", ElasticNet(random_state=SEED, max_iter=50000))]),
    "Ridge": Pipeline([("sc", StandardScaler()), ("m", Ridge())]),
    "BayesianRidge": Pipeline([("sc", StandardScaler()), ("m", BayesianRidge())]),
    "HuberReg": Pipeline([("sc", StandardScaler()), ("m", HuberRegressor(max_iter=1000))]),
    "KNN": Pipeline([("sc", StandardScaler()), ("m", KNeighborsRegressor())]),
    "RF": RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    "GBoost": GradientBoostingRegressor(random_state=SEED, loss="absolute_error"),
    "XGB": XGBRegressor(
        random_state=SEED,
        objective="reg:absoluteerror",
        eval_metric="mae",
        tree_method="hist",
        n_jobs=-1,
        verbosity=0,
    ),
}

for name, est in BASELINE_MODELS.items():
    t0 = time.time()
    est = clone(est)
    est.fit(X_ps_train, y_ps_train)
    pred = est.predict(X_ps_test)
    m = reg_metrics(y_ps_test, pred)
    rem_results.append(dict(model=name, stage="ml_baseline", time_s=round(time.time()-t0, 1), **m))

rem_df = pd.DataFrame(rem_results).sort_values(["MAE", "RMSE", "MdAE"]).reset_index(drop=True)
rem_df.to_csv(ART_DIR / "remaining_poststart_baseline_summary.csv", index=False)

print("Post-start baseline summary:")
display(rem_df)



## ML tuning on temporal validation

Чтобы сравнение было честнее, тюнинг идёт не по holdout, а по **последнему temporal validation внутри train**.


In [ ]:

# Temporal inner split for tuning
val_cut = int(len(X_ps_train) * 0.8)
X_fit = X_ps_train.iloc[:val_cut].copy()
y_fit = y_ps_train.iloc[:val_cut].copy()
X_val = X_ps_train.iloc[val_cut:].copy()
y_val = y_ps_train.iloc[val_cut:].copy()

print("Inner fit:", X_fit.shape, "Inner val:", X_val.shape)


In [ ]:

TUNE_MODELS = ["ElasticNet", "Ridge", "HuberReg", "GBoost", "XGB"]

def build_tuned_model(model_name, trial):
    if model_name == "ElasticNet":
        return Pipeline([("sc", StandardScaler()), ("m", ElasticNet(
            alpha=trial.suggest_float("alpha", 1e-4, 10.0, log=True),
            l1_ratio=trial.suggest_float("l1_ratio", 0.50, 0.999),
            selection=trial.suggest_categorical("selection", ["cyclic", "random"]),
            max_iter=50000,
            random_state=SEED,
        ))])

    if model_name == "Ridge":
        return Pipeline([("sc", StandardScaler()), ("m", Ridge(
            alpha=trial.suggest_float("alpha", 1e-2, 1e4, log=True)
        ))])

    if model_name == "HuberReg":
        return Pipeline([("sc", StandardScaler()), ("m", HuberRegressor(
            epsilon=trial.suggest_float("epsilon", 1.05, 2.5),
            alpha=trial.suggest_float("alpha", 1e-5, 10.0, log=True),
            max_iter=2000,
        ))])

    if model_name == "GBoost":
        loss = trial.suggest_categorical("loss", ["absolute_error", "huber"])
        params = {
            "loss": loss,
            "n_estimators": trial.suggest_int("n_estimators", 80, 320, step=20),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15, log=True),
            "max_depth": trial.suggest_int("max_depth", 1, 4),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 30),
            "subsample": trial.suggest_float("subsample", 0.60, 1.0),
            "max_features": trial.suggest_categorical("max_features", [None, 0.7, 0.85, 1.0]),
            "random_state": SEED,
        }
        if loss == "huber":
            params["alpha"] = trial.suggest_float("alpha", 0.80, 0.97)
        return GradientBoostingRegressor(**params)

    if model_name == "XGB":
        return XGBRegressor(
            random_state=SEED,
            objective="reg:absoluteerror",
            eval_metric="mae",
            tree_method="hist",
            n_jobs=-1,
            verbosity=0,
            n_estimators=trial.suggest_int("n_estimators", 100, 800, step=50),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.10, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 5),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
            subsample=trial.suggest_float("subsample", 0.50, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.50, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-2, 100.0, log=True),
            gamma=trial.suggest_float("gamma", 0.0, 5.0),
            max_bin=trial.suggest_int("max_bin", 128, 512, step=64),
        )

    raise ValueError(model_name)

N_TRIALS = {
    "ElasticNet": 40,
    "Ridge": 30,
    "HuberReg": 40,
    "GBoost": 60,
    "XGB": 60,
}

tuned_rows = []
tuned_models = {}

for model_name in TUNE_MODELS:
    print("=" * 90)
    print("Optuna |", model_name)
    print("=" * 90)

    # baseline on inner val for fair keep-or-replace
    baseline = clone(BASELINE_MODELS[model_name])
    baseline.fit(X_fit, y_fit)
    baseline_val_pred = baseline.predict(X_val)
    baseline_val_mae = mean_absolute_error(y_val, baseline_val_pred)

    def objective(trial):
        est = build_tuned_model(model_name, trial)
        est.fit(X_fit, y_fit)
        pred = est.predict(X_val)
        return float(mean_absolute_error(y_val, pred))

    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    t0 = time.time()
    study.optimize(objective, n_trials=N_TRIALS[model_name], show_progress_bar=True)
    elapsed = time.time() - t0

    tuned_est = build_tuned_model(model_name, optuna.trial.FixedTrial(study.best_params))
    tuned_est.fit(X_fit, y_fit)
    tuned_val_pred = tuned_est.predict(X_val)
    tuned_val_mae = mean_absolute_error(y_val, tuned_val_pred)

    if tuned_val_mae < baseline_val_mae:
        selected = tuned_est
        selected_from = "tuned"
        selected_val_mae = tuned_val_mae
    else:
        selected = baseline
        selected_from = "baseline_keep"
        selected_val_mae = baseline_val_mae

    # fit selected on full train and evaluate on holdout
    final_est = clone(selected)
    final_est.fit(X_ps_train, y_ps_train)
    pred_test = final_est.predict(X_ps_test)
    m = reg_metrics(y_ps_test, pred_test)

    tuned_models[model_name] = final_est
    tuned_rows.append({
        "model": model_name,
        "selected_from": selected_from,
        "baseline_val_MAE": round(float(baseline_val_mae), 4),
        "tuned_val_MAE": round(float(tuned_val_mae), 4),
        "final_val_MAE": round(float(selected_val_mae), 4),
        "holdout_poststart_MAE": round(m["MAE"], 4),
        "holdout_poststart_RMSE": round(m["RMSE"], 4),
        "holdout_poststart_MdAE": round(m["MdAE"], 4),
        "best_params": study.best_params,
        "time_s": round(elapsed, 1),
    })

tuned_df = pd.DataFrame(tuned_rows).sort_values(["holdout_poststart_MAE", "final_val_MAE"]).reset_index(drop=True)
tuned_df.to_csv(ART_DIR / "remaining_poststart_tuned_results.csv", index=False)
display(tuned_df)


In [ ]:

# Final selection table: naive + ML baseline + tuned ML
baseline_ml_df = rem_df[rem_df["stage"] == "ml_baseline"].copy().rename(columns={"MAE": "holdout_poststart_MAE", "RMSE": "holdout_poststart_RMSE", "MdAE": "holdout_poststart_MdAE"})
baseline_ml_df["selected_from"] = "baseline"
baseline_ml_df["final_val_MAE"] = np.nan
baseline_ml_df["best_params"] = None

baseline_naive_df = rem_df[rem_df["stage"] == "naive"].copy().rename(columns={"MAE": "holdout_poststart_MAE", "RMSE": "holdout_poststart_RMSE", "MdAE": "holdout_poststart_MdAE"})
baseline_naive_df["selected_from"] = "naive"
baseline_naive_df["final_val_MAE"] = np.nan
baseline_naive_df["best_params"] = None

final_df = pd.concat([
    baseline_naive_df[["model", "selected_from", "final_val_MAE", "holdout_poststart_MAE", "holdout_poststart_RMSE", "holdout_poststart_MdAE", "best_params"]],
    baseline_ml_df[["model", "selected_from", "final_val_MAE", "holdout_poststart_MAE", "holdout_poststart_RMSE", "holdout_poststart_MdAE", "best_params"]],
    tuned_df[["model", "selected_from", "final_val_MAE", "holdout_poststart_MAE", "holdout_poststart_RMSE", "holdout_poststart_MdAE", "best_params"]],
], ignore_index=True)

final_df = final_df.sort_values(["holdout_poststart_MAE", "holdout_poststart_RMSE"]).reset_index(drop=True)
final_df.to_csv(ART_DIR / "remaining_poststart_final_summary.csv", index=False)

print("Final summary (post-start):")
display(final_df.head(20))

# best tuned/classical model
tuned_or_baseline = final_df[~final_df["selected_from"].isin(["naive"])].copy()
best_model_name = tuned_or_baseline.iloc[0]["model"]
print("Best model family on post-start:", best_model_name)


In [ ]:

# Per-status breakdown for best tuned/classical model on post-start
if best_model_name in tuned_models:
    best_model = tuned_models[best_model_name]
else:
    best_model = clone(BASELINE_MODELS[best_model_name]).fit(X_ps_train, y_ps_train)

pred_ps = best_model.predict(X_ps_test)

status_rows = []
for status in STATUS_ORDER[1:]:
    mask = test_status_names_ps == status
    if mask.sum() == 0:
        continue
    mae_s = mean_absolute_error(y_ps_test.values[mask], pred_ps[mask])
    rmse_s = np.sqrt(mean_squared_error(y_ps_test.values[mask], pred_ps[mask]))
    mdae_s = median_absolute_error(y_ps_test.values[mask], pred_ps[mask])
    status_rows.append({
        "status": status,
        "n": int(mask.sum()),
        "mean_remaining": float(y_ps_test.values[mask].mean()),
        "MAE": float(mae_s),
        "RMSE": float(rmse_s),
        "MdAE": float(mdae_s),
    })

status_df = pd.DataFrame(status_rows)
status_df.to_csv(ART_DIR / "remaining_poststart_status_breakdown.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=120)

axes[0].barh(status_df["status"], status_df["MAE"], color="#90CAF9", edgecolor="#1565C0")
axes[0].set_xlabel("MAE (hours)")
axes[0].set_title(f"Per-status MAE ({best_model_name}, post-start)")
axes[0].invert_yaxis()
axes[0].grid(axis="x", alpha=0.3)

axes[1].barh(status_df["status"], status_df["mean_remaining"], color="#EF9A9A", edgecolor="#C62828")
axes[1].set_xlabel("Mean remaining (hours)")
axes[1].set_title("Mean remaining by status")
axes[1].invert_yaxis()
axes[1].grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

display(status_df)



## Supplementary: all statuses including backlog

Это не основной критерий, но позволяет честно показать, как модель ведёт себя по всей цепочке статусов, включая backlog.


In [ ]:

# Fit the same best model family on all snapshots and evaluate on all statuses
def rebuild_selected_model(model_name):
    if model_name in tuned_models:
        # tuned model params are already inside estimator; rebuild by clone and fit on all rows
        return clone(tuned_models[model_name])
    return clone(BASELINE_MODELS[model_name])

best_all_model = rebuild_selected_model(best_model_name)
best_all_model.fit(X_all_train, y_all_train)
pred_all = best_all_model.predict(X_all_test)

all_metrics = reg_metrics(y_all_test, pred_all)
print("Supplementary all-status metrics:")
print(all_metrics)

all_status_rows = []
for status in STATUS_ORDER:
    mask = test_status_names_all == status
    if mask.sum() == 0:
        continue
    all_status_rows.append({
        "status": status,
        "n": int(mask.sum()),
        "mean_remaining": float(y_all_test.values[mask].mean()),
        "MAE": float(mean_absolute_error(y_all_test.values[mask], pred_all[mask])),
        "RMSE": float(np.sqrt(mean_squared_error(y_all_test.values[mask], pred_all[mask]))),
        "MdAE": float(median_absolute_error(y_all_test.values[mask], pred_all[mask])),
    })

all_status_df = pd.DataFrame(all_status_rows)
all_status_df.to_csv(ART_DIR / "remaining_all_status_breakdown.csv", index=False)

display(all_status_df)


In [ ]:

interpretation = {
    "status_order_used": STATUS_ORDER,
    "primary_task": "predict remaining time on post-start snapshots (status_idx > 0)",
    "supplementary_task": "evaluate the same best model family on all statuses including backlog",
    "main_split_rule": "temporal split by task key, 80/20",
    "primary_winner": best_model_name,
    "notes": [
        "Merged position is corrected to be after Протестировано, per business order supplied by the user.",
        "Blocked state is excluded from the milestone chain.",
        "Current status duration is not used as a feature; snapshots are interpreted as the moment of entering the current status.",
        "Past statuses are known; current and future statuses are zeroed to avoid leakage.",
    ],
}
with open(ART_DIR / "remaining_status_experiment_summary.json", "w", encoding="utf-8") as f:
    json.dump(interpretation, f, ensure_ascii=False, indent=2)

print("Artifacts saved to:", ART_DIR.resolve())
print("Primary winner:", best_model_name)
